![WEB SCRAPING MIDDLE EAST COUNTRIES USED CARs Information](https://raw.githubusercontent.com/Owaiskhan9654/DigiGene/main/web-scraping-middle-east-Dataset22.png)

 ### This Notebook is Created by [**Owais Ahmad**](https://www.linkedin.com/in/owaiskhan9654/) for Fetching Dataset From API and do Some Data Exploratory on it
 ### Also I published the Dataset generated few months back[[Link](https://www.kaggle.com/datasets/owaiskhan9654/uae-car-used-dataset)] Also attached to this notebook. Now I am also Open Sourcing the code
 ### Note- This is for Tutorial/Research Purpose only
 
 
 ### Here I am Scraping data from https://www.drivearabia.com/ and this code is in sync with the website currently (December 2025). 
 ### Also remember this Repository is for you to get idea how it works and each website has different structure so it totally depends upon how you are writing the code and on the website from which you want Scrape the data. If you find it Useful Please do upvote the notebook
 
 
 
- **Email owaiskhan9654@gmail.com**
- **Contact +919515884381**


<center><h1 style = "font-size:25px;font-family: Comic Sans MS"> Please do upvote as I put lots of hard work in this. Also feel free to comment if you have any queries:)</h1></center>

# **<span style="color:#9f51e7;">Installing cloudscraper package</span>**

In [ ]:
!pip install -q cloudscraper

# **<span style="color:#9f51e7;">Some basics Import</span>**

In [ ]:
from bs4 import BeautifulSoup
import cloudscraper
from tqdm.notebook import tqdm
import pandas as pd
import re
import math

In [ ]:
%%capture

scraper = cloudscraper.create_scraper( browser={
        'browser': 'firefox',
        'platform': 'windows',
        'mobile': False
    }) # Cloud Scrapper Object

print(scraper.get("https://www.drivearabia.com/carprices/uae/").text)

In [ ]:
carbrands_slug = []
Website_links_all  = []
page_no = 1


for page_no in range(1,15):
    url = "https://www.drivearabia.com/carprices/uae/?page="+str(page_no)
    print(url)
    content=scraper.get(url).text
    soup = BeautifulSoup(content)
    brands=soup.find_all({"a":{"class":{"brand-item"}}})

    for i in brands:
        if i is not None and hasattr(i, "attrs") and "href" in i.attrs:
            href = i.attrs['href']
            lp = href.strip("/").split("/")[-1]
            if lp.isdigit():  
                # print(href)
                carbrands_slug.append(href)
                url="https://www.drivearabia.com{}".format(href)
                Website_links_all.append(url)

    page_no += 1


<center><h1 style = "font-size:25px;font-family: Comic Sans MS"> Out of 8115 websites I am downloading 2000 samples on Kaggle Kernal since it is crashing.</h1></center>
<center><h1 style = "font-size:25px;font-family: Comic Sans MS">Please remove the condition of break loop if you want to scrape all the data locally</h1></center>

In [ ]:
from tqdm import tqdm

origin_country=[]
Approx_cost=[]
BodyType=[]
additionalType=[]
weight=[]
Engine=[]
Gearbox=[]
Power=[]
Torque=[]
Fuel_Econ_L_100km=[]
Fuel_Econ_km_L=[]
sale_country=[]
Performance_0_100=[]
Top_speed_kph=[]
Overview=[]
Manufacturer=[]
Brand=[]
Model=[]

for j,i in tqdm(enumerate(Website_links_all)):
    scraper = cloudscraper.create_scraper(  browser={
        'browser': 'firefox',
        'platform': 'windows',
        'mobile': False
    }) 
    content=scraper.get(i).text
    soup = BeautifulSoup(content, 'html.parser')

    
    section = soup.find("section", {"id": "trim-prices"})
    avg_final_price = None
    try:
        if section:
            rows = section.find_all("div", class_="flex border-b border-zinc-4 min-w-0")
            prices = []
            for row in rows:
                cols = row.find_all("div", recursive=False)
                if len(cols) == 2:
                    price_text = cols[1].get_text(strip=True)
                    nums = [int(x.replace(',', '')) for x in re.findall(r'\d[\d,]*', price_text)]
                    if len(nums) == 1:
                        prices.append(nums[0])
                    elif len(nums) == 2:
                        prices.append((nums[0] + nums[1]) / 2)  # take mid of range
        
            avg_final_price = sum(prices) / len(prices)
            # print(f"Average Final Price: {avg_final_price:.0f} AED")
            Approx_cost.append(math.ceil(avg_final_price))
                
    except:
        Approx_cost.append('NA')
        
    try:

        row = soup.find("div", class_="flex border-b border-zinc-4 min-w-0", string=None)
        for row in soup.find_all("div", class_="flex border-b border-zinc-4 min-w-0"):
            label = row.find("span")
            if label and "Brand Origin" in label.get_text(strip=True):
                value_div = row.find_all("div")[-1]
                origin_country.append(value_div.get_text(strip=True))
            break  # only one Brand Origin row expected
        
        # print(origin_country)

    except:
        origin_country.append('NA')       
        
    try:
        for row in soup.find_all("div", class_="flex border-b border-zinc-4 min-w-0"):
            label = row.find("span")
            if label and "Body Type" in label.get_text(strip=True):
                value_div = row.find_all("div")[-1]
                BodyType.append(value_div.get_text(strip=True))
            break  # only one Body Type row expected
        
        # print(BodyType)
    except:
        BodyType.append('NA')
                
    try:
        weight.append(soup.find("td",{'dir':'ltr','itemprop':"weight"}).contents[0])
    except: 
        weight.append('NA')   
    
    try:
        sale_country.append(i[i.index('carprices/')+len('carprices/'):i.find('/',i.index('carprices/')+len('carprices/'))])
    except:
        sale_country.append('NA')        
    
    try:        
        for row in soup.find_all("div", class_="flex border-b border-zinc-4 min-w-0"):
            label = row.find("span")
            if label and "Engine Type" in label.get_text(strip=True):
                value_div = row.find_all("div")[-1]
                Engine.append(value_div.get_text(strip=True))
            break
        
        # print(Engine)        
    except:
        Engine.append('NA')
        
    try:
        Gearbox.append(soup.find("td",{'itemprop':"vehicleTransmission"}).contents[0])
    except:
        Gearbox.append('NA')
        
    try:
        Power.append(soup.find("span",{'itemprop':"enginePower"}).contents[0])
    except:
        Power.append('NA')
        
    try:
        Torque.append(soup.find("span",{'itemprop':"torque"}).contents[0])
    except:
        Torque.append('NA')
        
    try:
        Fuel_Econ_L_100km.append(soup.find("td",{'itemprop':"fuelEfficiency"}).contents[0])
    except:
        Fuel_Econ_L_100km.append('NA')
        
    try:
        Fuel_Econ_km_L.append(soup.find_all("td",{'itemprop':"fuelEfficiency"})[1].contents[0])
    except:
        Fuel_Econ_km_L.append('NA')
    
    try:
        Performance_0_100.append(soup.find("td",{'itemprop':"speed"}).contents[0])
    except:
        Performance_0_100.append('NA')
    
    try:
        Top_speed_kph.append(soup.find_all("td",{'itemprop':"speed"})[1].contents[0])
    except:    
        Top_speed_kph.append('NA')
    
    try:
        Overview.append(soup.find("div",{'class':"vehicle-overview"}).text.replace('\n',''))
    except:
        Overview.append('NA')
        
    try:
        Manufacturer.append(soup.find("span",{'itemprop':"manufacturer"}).contents[0])
    except:
        Manufacturer.append('NA')
        
    try:
        Brand.append(soup.find("span",{'itemprop':"brand"}).contents[0])
    except:
        Brand.append('NA')
        
    try:
        Model.append(soup.find("span",{'itemprop':"model"}).contents[0])
    except:
        Model.append('NA')
        
    j=j+1
    if j==1000:
        break


In [ ]:
from tqdm import tqdm
import re, math, cloudscraper
from bs4 import BeautifulSoup

origin_country, Approx_cost, BodyType, additionalType, weight, Engine, Gearbox, Power, Torque = [],[],[],[],[],[],[],[],[]
Fuel_Econ_L_100km, Fuel_Econ_km_L, sale_country, Performance_0_100, Top_speed_kph = [],[],[],[],[]
Overview, Manufacturer, Brand, Model = [],[],[],[]

for j, i in tqdm(enumerate(Website_links_all)):
    scraper = cloudscraper.create_scraper(browser={'browser': 'firefox','platform': 'windows','mobile': False})
    content = scraper.get(i).text
    soup = BeautifulSoup(content, 'html.parser')

    # ---- Average price ----
    try:
        section = soup.find("section", {"id": "trim-prices"})
        if section:
            prices = []
            for row in section.find_all("div", class_="flex border-b border-zinc-4 min-w-0"):
                cols = row.find_all("div", recursive=False)
                if len(cols) == 2:
                    nums = [int(x.replace(',', '')) for x in re.findall(r'\d[\d,]*', cols[1].get_text(strip=True))]
                    if len(nums) == 1:
                        prices.append(nums[0])
                    elif len(nums) == 2:
                        prices.append((nums[0]+nums[1])/2)
            avg_final_price = sum(prices)/len(prices)
            Approx_cost.append(math.ceil(avg_final_price))
        else:
            Approx_cost.append('NA')
    except:
        Approx_cost.append('NA')

    specs = {}
    for row in soup.select("#specs div.flex.border-b.border-zinc-4"):
        cols = row.find_all("div", recursive=False)
        if len(cols) >= 2:
            label = cols[0].get_text(strip=True)
            value = cols[1].get_text(strip=True)
            specs[label] = value


    
    # ---- All specs in one pass ----
    # specs = {}
    for row in soup.find_all("div", class_="flex border-b border-zinc-4 min-w-0"):
        label_span = row.find("span")
        if label_span:
            label = label_span.get_text(strip=True)
            value = row.find_all("div")[-1].get_text(strip=True)
            specs[label] = value




    
    origin_country.append(specs.get("Brand Origin", 'NA'))
    
    BodyType.append(
        specs.get("Body Type", specs.get("Body Style", 'NA'))
    )
    
    Engine.append(specs.get("Engine Type", 'NA'))
    weight.append(specs.get("Weight", 'NA'))
    
    Gearbox.append(specs.get("Transmission", 'NA'))          # WILL ALWAYS BE NA
    Power.append(specs.get("Power", 'NA'))                  # WILL ALWAYS BE NA
    Torque.append(specs.get("Torque", 'NA'))                # WILL ALWAYS BE NA
    
    Fuel_Econ_L_100km.append(
        specs.get("Fuel Economy (L/100km)", 'NA')
    )
    
    Fuel_Econ_km_L.append(
        specs.get("Fuel Economy (km/L)", specs.get("Fuel Economy", 'NA'))
    )
    
    Performance_0_100.append(specs.get("0-100 km/h", 'NA'))
    Top_speed_kph.append(specs.get("Top Speed", 'NA'))
    
    sale_country.append(
        i[i.index('carprices/')+len('carprices/'):i.find('/',i.index('carprices/')+len('carprices/'))]
        if 'carprices/' in i else 'NA'
    )
    
    Overview.append(
        soup.find("div", {'class': "vehicle-overview"}).get_text(strip=True)
        if soup.find("div", {'class': "vehicle-overview"}) else 'NA'
    )
    
    Manufacturer.append(specs.get("Manufactured In", 'NA'))
    Brand.append(specs.get("Brand Origin", 'NA'))            # YOU USED WRONG KEY
    Model.append(specs.get("Vehicle", 'NA'))                 # BEST POSSIBLE VALUE
        

    if j==1000:
        break


In [ ]:
print(len(Website_links_all[0:1000]))
print("Approx_cost: ",len(Approx_cost))
print("origin_country: ",len(origin_country))
print("sale_country: ", len(sale_country))
print("Manufacturer: ",len(Manufacturer))
print("Brand: ",len(Brand))
print("Model: ",len(Model))
print("BodyType: ",len(BodyType))
print("weight: ", len(weight))
print("Gearbox: ",len(Gearbox))
print("Power: ",len(Power))
print("Torque: ",len(Torque))
print("Fuel_Econ_L_100km: ", len(Fuel_Econ_L_100km))
print("Fuel_Econ_km_L: ",len(Fuel_Econ_km_L))
print("Performance_0_100: ",len(Performance_0_100))
print("Top_speed_kph: ",len(Top_speed_kph))
print("Overview: ",len(Overview))


# **<span style="color:#9f51e7;">Parsing all the data into DataFrame and then exporting to csv</span>**

In [ ]:
df=pd.DataFrame({'Link':Website_links_all[0:1000],'Approx Cost':Approx_cost,'Origin Country':origin_country,\
                 'Sale Country':sale_country,'Body Type':BodyType,'Weight':weight,\
                 #'Additional Type':additionalType,\
                 'Manufacturer':Manufacturer,'Brand':Brand,'Model Year':Model,\
                 'Gear box':Gearbox,'Power (hp)':Power,'Torque (Nm)':Torque,\
                 'Fuel Econ (L/100km)':Fuel_Econ_L_100km,'Fuel Econ (km/L)':Fuel_Econ_km_L,\
                 'Performance 0-100 kph (sec)':Performance_0_100,'Top speed (kph)':Top_speed_kph,\
                 'Overview':Overview})

In [ ]:
df

In [ ]:
df.to_csv('All_UAE_used_Car_Dataset_07_July_2022.csv',index=False)

<center><h1 style = "font-size:30px;font-family: Comic Sans MS">Similarly We can Fetch Dataset for all the GCC countries</h1></center>

<center><h1 style = "font-size:25px;font-family: Comic Sans MS"> Please do upvote as I put lots of hard work in this. Also feel free to comment down if you have any queries:)</h1></center>

 ### This Notebook is Created by [**Owais Ahmad**](https://www.linkedin.com/in/owaiskhan9654/) for Fetching Dataset using Web scraping and also to do Some Data Exploratory on it
 ### Note-: This is for Research Purpose only and I don't have any responsibility of it from legal point of view
 ## Also I published the Dataset generated few months back[[Link](https://www.kaggle.com/datasets/owaiskhan9654/uae-car-used-dataset)]. Now I am also Open Sourcing the code also
 
 
 
- **Email owaiskhan9654@gmail.com**
- **Contact +919515884381**

<center><h1 style = "font-size:28px;font-family: Comic Sans MS"> I am thinking to build a machine learning model which will be able to predict cost price of the used vehical based on Year of Manufacturing, Country of origin, selling country and many other similar features</h1></center>

In [ ]:
# Used Car Price Prediction Model
# Building ML model to predict car prices based on various features

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------
# Load the dataset
# ---------------------------------------------------
df = pd.read_csv('All_UAE_used_Car_Dataset_07_July_2022.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

print("\nDataset info:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

# ---------------------------------------------------
# Data Cleaning and Preprocessing
# ---------------------------------------------------
def clean_and_preprocess_data(df):
    """
    Clean and preprocess the car dataset
    """
    data = df.copy()
    
    # ---------- Clean target: Approx Cost ----------
    def clean_price(price_str):
        if pd.isna(price_str) or price_str == 'NA':
            return np.nan
        
        price_str = str(price_str).replace('AED', '').replace(',', '').strip()
        
        # Handle ranges "10000 - 15000"
        if '-' in price_str:
            parts = price_str.split('-')
            if len(parts) == 2:
                try:
                    low = float(parts[0].strip())
                    high = float(parts[1].strip())
                    return (low + high) / 2
                except:
                    return np.nan
        
        # Single values
        try:
            return float(price_str)
        except:
            return np.nan
    
    data['Price'] = data['Approx Cost'].apply(clean_price)
    
    # ---------- Universal numeric extractor ----------
    def extract_any_numeric(col):
        """
        Take a pandas Series and pull the first number from each cell.
        Works with values like '250 hp', '0-100 in 5.5 sec', '101 kph', etc.
        """
        col = col.astype(str)
        s = col.str.extract(r'(\d+\.?\d*)', expand=False)
        return pd.to_numeric(s, errors='coerce')
    
    def safe_numeric_feature(data, pattern_list):
        """
        Find the first column whose name matches any regex in pattern_list
        and extract numeric values from it.
        If nothing matches, return all-NaN series.
        """
        regex = '|'.join(pattern_list)
        cols = data.filter(regex=regex, axis=1)
        if cols.shape[1] == 0:
            # no matching column
            return pd.Series(np.nan, index=data.index, dtype=float)
        return extract_any_numeric(cols.iloc[:, 0])
    
    # ---------- Engine / performance / weight (auto-detected) ----------
    # Power (hp, BHP, etc.)
    data['Power_hp'] = safe_numeric_feature(
        data,
        [r'(?i)power', r'(?i)hp', r'(?i)bhp']
    )
    
    # Torque (Nm, N.m)
    data['Torque_Nm'] = safe_numeric_feature(
        data,
        [r'(?i)torque', r'(?i)nm']
    )
    
    # Weight / kerb weight
    data['Weight_kg'] = safe_numeric_feature(
        data,
        [r'(?i)weight', r'(?i)kerb']
    )
    
    # Fuel economy L/100km (if only km/L available, you can convert later if needed)
    data['Fuel_L_100km'] = safe_numeric_feature(
        data,
        [r'(?i)fuel', r'(?i)econ', r'(?i)consumption']
    )
    
    # Performance 0-100 kph
    data['Performance_0_100'] = safe_numeric_feature(
        data,
        [r'(?i)0.?100', r'(?i)0-100', r'(?i)accel']
    )
    
    # Top speed
    data['Top_Speed'] = safe_numeric_feature(
        data,
        [r'(?i)top', r'(?i)speed', r'(?i)kph']
    )
    
    # ---------- Model year and car age ----------
    data['Model_Year'] = safe_numeric_feature(
        data,
        [r'(?i)model.*year', r'(?i)year']
    )
    data['Car_Age'] = 2025 - data['Model_Year']
    
    # ---------- Categorical cleaning ----------
    categorical_cols = [
        'Origin Country', 
        'Sale Country', 
        'Manufacturer', 
        'Brand', 
        'Body Type'
    ]
    
    for col in categorical_cols:
        if col not in data.columns:
            data[col] = 'Unknown'
        data[col] = data[col].fillna('Unknown').astype(str)
        data[col] = data[col].replace('NA', 'Unknown')
    
    return data

# Apply preprocessing
data_clean = clean_and_preprocess_data(df)

# Remove rows with missing target variable
data_clean = data_clean.dropna(subset=['Price'])

# Remove extreme outliers in price
price_q1 = data_clean['Price'].quantile(0.01)
price_q99 = data_clean['Price'].quantile(0.99)
data_clean = data_clean[(data_clean['Price'] >= price_q1) & (data_clean['Price'] <= price_q99)]

print(f"\nAfter cleaning: {data_clean.shape[0]} records")
print(f"Price range: {data_clean['Price'].min():.0f} - {data_clean['Price'].max():.0f} AED")

# ---------------------------------------------------
# Exploratory Data Analysis
# ---------------------------------------------------
plt.figure(figsize=(15, 10))

# 1) Price distribution
plt.subplot(2, 3, 1)
plt.hist(data_clean['Price'], bins=50, edgecolor='black', alpha=0.7)
plt.title('Price Distribution')
plt.xlabel('Price (AED)')
plt.ylabel('Frequency')

# 2) Price vs Car Age
plt.subplot(2, 3, 2)
if 'Car_Age' in data_clean.columns:
    valid_age = data_clean.dropna(subset=['Car_Age'])
    if not valid_age.empty:
        plt.scatter(valid_age['Car_Age'], valid_age['Price'], alpha=0.5)
    plt.title('Price vs Car Age')
    plt.xlabel('Car Age (years)')
    plt.ylabel('Price (AED)')
else:
    plt.text(0.5, 0.5, 'Car_Age not available', ha='center')

# 3) Top brands by count
plt.subplot(2, 3, 3)
top_brands = data_clean['Brand'].value_counts().head(10)
top_brands.plot(kind='bar')
plt.title('Top 10 Car Brands')
plt.xlabel('Brand')
plt.ylabel('Count')
plt.xticks(rotation=45)

# 4) Body type distribution
plt.subplot(2, 3, 4)
body_types = data_clean['Body Type'].value_counts().head(8)
body_types.plot(kind='bar')
plt.title('Body Type Distribution')
plt.xlabel('Body Type')
plt.ylabel('Count')
plt.xticks(rotation=45)

# 5) Price vs Engine Power
plt.subplot(2, 3, 5)
valid_power = data_clean.dropna(subset=['Power_hp'])
if not valid_power.empty:
    plt.scatter(valid_power['Power_hp'], valid_power['Price'], alpha=0.5)
plt.title('Price vs Engine Power')
plt.xlabel('Power (hp)')
plt.ylabel('Price (AED)')

# 6) Correlation matrix
plt.subplot(2, 3, 6)
numerical_for_corr = [
    'Price', 'Car_Age', 'Power_hp', 'Torque_Nm', 
    'Weight_kg', 'Performance_0_100', 'Top_Speed', 'Fuel_L_100km'
]
corr_cols = [c for c in numerical_for_corr if c in data_clean.columns]
corr_data = data_clean[corr_cols].corr()
sns.heatmap(corr_data, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Feature Engineering and Model Building
# ---------------------------------------------------
def prepare_features_for_modeling(data):
    """
    Prepare features for machine learning
    """
    numerical_features = [
        'Car_Age',
        'Power_hp',
        'Torque_Nm',
        'Weight_kg',
        'Performance_0_100',
        'Top_Speed',
        'Fuel_L_100km'
    ]
    
    categorical_features = [
        'Origin Country',
        'Sale Country',
        'Brand',
        'Body Type'
    ]
    
    # Ensure all feature columns exist
    for col in numerical_features:
        if col not in data.columns:
            data[col] = np.nan
    for col in categorical_features:
        if col not in data.columns:
            data[col] = 'Unknown'
    
    X = data[numerical_features + categorical_features].copy()
    y = data['Price'].copy()
    
    return X, y, numerical_features, categorical_features

X, y, numerical_features, categorical_features = prepare_features_for_modeling(data_clean)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Replace inf with NaN (just in case)
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# ---------------------------------------------------
# Preprocessing pipelines
# ---------------------------------------------------
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# ---------------------------------------------------
# Define models
# ---------------------------------------------------
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Lasso Regression':   Lasso(alpha=1.0),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    cv_scores = cross_val_score(
        pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error'
    )
    cv_mae = -cv_scores.mean()
    
    results[name] = {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'CV_MAE': cv_mae,
        'Pipeline': pipeline
    }
    
    print(f"\n{name}:")
    print(f"  MAE: {mae:.2f} AED")
    print(f"  RMSE: {rmse:.2f} AED")
    print(f"  R²: {r2:.3f}")
    print(f"  CV MAE: {cv_mae:.2f} AED")

# ---------------------------------------------------
# Best model
# ---------------------------------------------------
best_model_name = min(results.keys(), key=lambda x: results[x]['MAE'])
best_pipeline = results[best_model_name]['Pipeline']

print(f"\nBest model: {best_model_name}")
print(f"Test MAE: {results[best_model_name]['MAE']:.2f} AED")

# ---------------------------------------------------
# Hyperparameter tuning (if tree-based)
# ---------------------------------------------------
if best_model_name in ['Random Forest', 'Gradient Boosting']:
    print(f"\nTuning hyperparameters for {best_model_name}...")
    
    if best_model_name == 'Random Forest':
        param_grid = {
            'regressor__n_estimators': [100, 200],
            'regressor__max_depth': [10, 20, None],
            'regressor__min_samples_split': [2, 5],
            'regressor__min_samples_leaf': [1, 2]
        }
    else:  # Gradient Boosting
        param_grid = {
            'regressor__n_estimators': [100, 200],
            'regressor__max_depth': [3, 5],
            'regressor__learning_rate': [0.05, 0.1],
            'regressor__subsample': [0.8, 1.0]
        }
    
    grid_search = GridSearchCV(
        best_pipeline,
        param_grid,
        cv=3,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    tuned_model = grid_search.best_estimator_
    y_pred_tuned = tuned_model.predict(X_test)
    
    mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
    rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
    r2_tuned = r2_score(y_test, y_pred_tuned)
    
    print(f"\nTuned {best_model_name}:")
    print(f"  MAE: {mae_tuned:.2f} AED")
    print(f"  RMSE: {rmse_tuned:.2f} AED")
    print(f"  R²: {r2_tuned:.3f}")
    print(f"  Best parameters: {grid_search.best_params_}")
    
    final_model = tuned_model
else:
    final_model = best_pipeline

# ---------------------------------------------------
# Feature importance (for tree-based models)
# ---------------------------------------------------
if hasattr(final_model.named_steps['regressor'], 'feature_importances_'):
    cat_ohe = final_model.named_steps['preprocessor'] \
                         .named_transformers_['cat'] \
                         .named_steps['onehot']
    
    ohe_feature_names = list(cat_ohe.get_feature_names_out(categorical_features))
    feature_names = numerical_features + ohe_feature_names
    
    importances = final_model.named_steps['regressor'].feature_importances_
    
    # feature_importance = pd.DataFrame({
    #     'feature': feature_names,
    #     'importance': importances
    # }).sort_values('importance', ascending=False)
    # ---------------------------------------------------
# Feature importance (for tree-based models)
# ---------------------------------------------------
if hasattr(final_model.named_steps['regressor'], 'feature_importances_'):
    reg = final_model.named_steps['regressor']
    pre = final_model.named_steps['preprocessor']

    # 1. Get feature names from ColumnTransformer (most robust)
    try:
        feature_names = pre.get_feature_names_out()
    except AttributeError:
        # fallback for older sklearn
        cat_ohe = pre.named_transformers_['cat'].named_steps['onehot']
        ohe_feature_names = cat_ohe.get_feature_names_out(categorical_features)
        feature_names = np.array(list(numerical_features) + list(ohe_feature_names))

    importances = reg.feature_importances_

    # 2. Make sure lengths match (avoid ValueError: All arrays must be of the same length)
    n = min(len(feature_names), len(importances))
    feature_names = feature_names[:n]
    importances = importances[:n]

    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)

    print("\nTop 15 Most Important Features:")
    print(feature_importance.head(15))

    # Plot feature importance
    plt.figure(figsize=(10, 8))
    top_features = feature_importance.head(15)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance')
    plt.title('Top 15 Feature Importances')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
# ---------------------------------------------------
# Prediction helper
# ---------------------------------------------------
def predict_car_price(
    model,
    origin_country,
    sale_country,
    brand,
    body_type,
    car_age,
    power_hp,
    torque_nm=None,
    weight_kg=None,
    performance_0_100=None,
    top_speed=None,
    fuel_l_100km=None
):
    """
    Predict car price for given specifications.
    Columns exactly match:
    Car_Age, Power_hp, Torque_Nm, Weight_kg, Performance_0_100, Top_Speed, 
    Fuel_L_100km, Origin Country, Sale Country, Brand, Body Type
    """
    input_data = pd.DataFrame({
        'Car_Age': [car_age],
        'Power_hp': [power_hp],
        'Torque_Nm': [torque_nm],
        'Weight_kg': [weight_kg],
        'Performance_0_100': [performance_0_100],
        'Top_Speed': [top_speed],
        'Fuel_L_100km': [fuel_l_100km],
        'Origin Country': [origin_country],
        'Sale Country': [sale_country],
        'Brand': [brand],
        'Body Type': [body_type]
    })
    
    prediction = model.predict(input_data)[0]
    return prediction

# ---------------------------------------------------
# Example prediction
# ---------------------------------------------------
print("\n" + "="*50)
print("EXAMPLE PREDICTION")
print("="*50)

example_price = predict_car_price(
    final_model,
    origin_country='Germany',
    sale_country='uae',
    brand='BMW',
    body_type='Compact SUV',
    car_age=3,
    power_hp=250,
    torque_nm=350,
    weight_kg=1600,
    performance_0_100=6.5,
    top_speed=240,
    fuel_l_100km=8.5
)

print(f"Predicted price for a 3-year-old BMW Compact SUV with 250hp: {example_price:.0f} AED")
print("\nModel saved as 'final_model' variable for future predictions!")
